### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Hello, how are you?")
response.content

c:\Users\matcg\OneDrive\Documents\GitHub\LangChain_Updated\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


"<think>\nOkay, the user asked Hello, how are you? I should respond with a friendly greeting and offer assistance. I need to keep it natural and conversational. Let me just check if there's anything else I need to consider. No, that should cover it. Let me put it all together in a warm and welcoming way.\n</think>\n\nHi there! I'm doing great, thanks for asking. How can I assist you today? 😊"

In [14]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # In a real implementation, this would call a weather API.
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools =model.bind_tools([get_weather]) 
# Uma outra forma de usar o tool dentro de um modelo é pela criacao de agentes, como fizemos anteriormente no arquivo 1-langchainintro.ipynb.

In [15]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"arguments: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with "Boston" as the location. Make sure the JSON is correctly formatted with the function name and arguments. Let me double-check the syntax to avoid any errors.\n', 'tool_calls': [{'id': 'majehyytc', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 156, 'total_tokens': 265, 'completion_time': 0.182322394, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.008079746, 'prompt_tokens_details': None, 'queue_time': 0.191812524, 'total_time': 0.19040214}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_dema

Outra forma, visto anteriormente

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)

response = agent.invoke({"messages": [{"role": "user", "content": "What's the weather like in Boston??"}]})
print(response)

{'messages': [HumanMessage(content="What's the weather like in Boston??", additional_kwargs={}, response_metadata={}, id='82dcb68a-d701-45ac-b6e1-d91f7017858b'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is provided as Boston. So I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'mkv9pkx8j', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 162, 'total_tokens': 250, 'completion_time': 0.143515091, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.006489706, 'prompt_tokens_details': None, 'queue_time': 0.190669975, 'total_time': 0.150004797}, 'model_name': 'qwen/qwe

### Tool Execution Loops

In [17]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
print(ai_msg)
print(messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    print(f"Tool result: {tool_result}")
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': '35pmkvjgv', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 155, 'total_tokens': 259, 'completion_time': 0.170367941, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.007694696, 'prompt_tokens_details': None, 'queue_time': 0.190815765, 'total_time': 0.178062637}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reaso

In [18]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': '35pmkvjgv', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 155, 'total_tokens': 259, 'completion_time': 0.170367941, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.007694696, 'prompt_tokens_details': None, 'queue_time': 0.190815765, 'total_time': 0.178062637}, 'model_name': 'qwen/qwen3-32b', 'syst